# Fine-Tuning with AutoTrain Advanced (LoRA & QLoRA)

**AutoTrain Advanced** is HuggingFace's no-code/low-code fine-tuning platform.
It has two modes:

| Mode | How | Best for |
|------|-----|----------|
| **UI** | Browser (`autotrain app`) | Quick experiments, no code |
| **CLI / Python** | `autotrain llm ...` | Reproducible runs, more control |

This notebook demonstrates the **CLI and Python** approach, which is more
scriptable and exposes options that the UI does not.

**LoRA vs QLoRA:** controlled by the `--quantization` flag.
- No flag → standard LoRA (float16 base)
- `--quantization int4` → QLoRA (4-bit NF4 base + LoRA adapters)
- `--quantization int8` → 8-bit LoRA

## Environment Setup

> **Critical: Python ==3.10.x is required.**
> AutoTrain Advanced does not support Python 3.11+. The `autotrain_ui`
> uv project already pins `requires-python = '==3.10.*'`.

Run once in a terminal:

```bash
cd projects/autotrain_ui
uv sync --no-install-workspace
bash setup.sh                     # downloads NLTK punkt tokenizer
uv run ipython kernel install --user --env VIRTUAL_ENV ../../.venv --name=autotrain_ui
```

Select the **`autotrain_ui`** kernel in JupyterLab.

In [ ]:
import sys, subprocess, os, pathlib, csv

print(f'Python {sys.version}')

# AutoTrain requires exactly Python 3.10.x
if sys.version_info[:2] != (3, 10):
    raise RuntimeError(
        f'AutoTrain requires Python 3.10.x, got {sys.version_info[:2]}. '
        'Use the autotrain_ui kernel.'
    )

import autotrain
print(f'AutoTrain version: {autotrain.__version__}')

result = subprocess.run(['autotrain', '--help'], capture_output=True, text=True)
print(f'AutoTrain CLI available: {result.returncode == 0}')

## HuggingFace Token — Non-Negotiable

> **Gotcha #1 — AutoTrain silently fails without a valid `HF_TOKEN`.**
> Unlike TRL/Unsloth which work fine for public models without auth,
> AutoTrain uses the token for dataset operations and model saving.
> Without it:
> - Training may start but produce empty or corrupt outputs
> - Model push fails silently
> - Some dataset ops return confusing 401 errors
>
> Also set `HUGGING_FACE_HUB_TOKEN` — AutoTrain checks both env vars
> depending on the version.

In [ ]:
import sys, pathlib, os
sys.path.insert(0, str(pathlib.Path('../../../').resolve()))
from dotenv_config import dot_env_settings

if not dot_env_settings.HF_TOKEN:
    raise EnvironmentError(
        'HF_TOKEN not set. Edit .env in the repo root '
        '(copy from .env.example) and add your token from '
        'https://huggingface.co/settings/tokens'
    )

os.environ['HF_TOKEN'] = dot_env_settings.HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = dot_env_settings.HF_TOKEN  # legacy name
print('HF_TOKEN set.')

## Dataset Column Names: Strict Requirements

> **Gotcha #2 — AutoTrain has exact column name requirements.**
> The column names are **not configurable** for the default task types:

| Task (`--trainer`) | Required columns |
|--------------------|-----------------|
| `sft` | `text` |
| `dpo` | `prompt`, `chosen`, `rejected` |
| `reward` | `text`, `rejected` |
| `orpo` | `prompt`, `chosen`, `rejected` |

For SFT, columns named `prompt`, `instruction`, `response`, or `output` are
**silently ignored**. Your data must have a single `text` column containing
the fully formatted prompt+response string.

In [ ]:
ALPACA_PROMPT = '### Instruction:\n{}\n\n### Response:\n{}'

examples = [
    {'instruction': 'What is machine learning?',
     'response': 'Machine learning is a field of AI where systems learn patterns from data rather than being explicitly programmed.'},
    {'instruction': 'Write a Python function to check if a number is prime.',
     'response': 'def is_prime(n):\n    if n < 2: return False\n    for i in range(2, int(n**0.5)+1):\n        if n % i == 0: return False\n    return True'},
    {'instruction': 'Explain the water cycle in simple terms.',
     'response': 'Water evaporates from oceans and lakes, rises as vapour, condenses into clouds, then falls as rain or snow, and flows back to the sea.'},
    {'instruction': 'What does HTTP stand for?',
     'response': 'HTTP stands for HyperText Transfer Protocol — the foundation of data communication on the web.'},
    {'instruction': 'Translate to Spanish: Good morning, have a nice day.',
     'response': 'Buenos días, que tengas un buen día.'},
    {'instruction': 'List three benefits of exercise.',
     'response': '1. Improves cardiovascular health. 2. Boosts mood via endorphin release. 3. Strengthens muscles and bones.'},
    {'instruction': 'What is the speed of light?',
     'response': 'The speed of light in a vacuum is approximately 299,792,458 metres per second (≈ 3×10⁸ m/s).'},
    {'instruction': 'Write a haiku about rain.',
     'response': 'Drops tap the window / Puddles swallow fallen leaves / The earth exhales grey'},
    {'instruction': 'What is the difference between RAM and ROM?',
     'response': 'RAM (Random Access Memory) is volatile working memory cleared on power-off. ROM (Read-Only Memory) is non-volatile and stores permanent firmware.'},
    {'instruction': 'How do I reverse a list in Python?',
     'response': 'Use the slice notation: reversed_list = my_list[::-1], or call my_list.reverse() to reverse in place.'},
]

# Format into a single 'text' column (required by AutoTrain SFT)
formatted = [
    {'text': ALPACA_PROMPT.format(ex['instruction'], ex['response'])}
    for ex in examples
]

data_dir = pathlib.Path('./autotrain_data')
data_dir.mkdir(exist_ok=True)
train_path = data_dir / 'train.csv'

with open(train_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['text'])
    writer.writeheader()
    writer.writerows(formatted)

print(f'Wrote {len(formatted)} examples to {train_path}')
print('\nFirst example:')
print(formatted[0]['text'])

## Key CLI Flags for `autotrain llm`

| Flag | Description |
|------|-------------|
| `--model` | HuggingFace model ID (must be **exact**) |
| `--data-path` | Path to folder containing `train.csv` |
| `--train-split` | Filename without extension (default: `train`) |
| `--project-name` | Output directory name (created in CWD) |
| `--text-column` | Column name (`text` for SFT) |
| `--trainer` | `sft`, `dpo`, `orpo`, `reward` |
| `--peft` | Enable LoRA (PEFT) adapters |
| `--lora-r` | LoRA rank |
| `--lora-alpha` | LoRA alpha scaling |
| `--quantization` | `int4` (QLoRA), `int8` (8-bit LoRA), or omit (standard LoRA) |
| `--lr` | Learning rate |
| `--batch-size` | Per-device batch size |
| `--epochs` | Number of training epochs |

> **Gotcha #3 — Model names must be exact HuggingFace IDs.**
> `'gpt2'` works. `'GPT-2'` does not. `'facebook/opt-125m'` works.
> AutoTrain validates nothing at config time — a wrong name causes a 404
> from the HF Hub at runtime, often with a confusing error message.

> **Gotcha #4 — AutoTrain controls the output directory.**
> The `--project-name` value becomes the directory name, created in your
> current working directory. You cannot redirect it. Plan your CWD before
> running.

In [ ]:
MODEL_NAME = 'gpt2'          # exact HF model ID
DATA_PATH = str(data_dir.resolve())
PROJECT_LORA = 'autotrain_gpt2_lora'
PROJECT_QLORA = 'autotrain_gpt2_qlora'

# Standard LoRA command
cmd_lora = [
    'autotrain', 'llm',
    '--model', MODEL_NAME,
    '--data-path', DATA_PATH,
    '--train-split', 'train',
    '--project-name', PROJECT_LORA,
    '--text-column', 'text',
    '--trainer', 'sft',
    '--peft',
    '--lora-r', '8',
    '--lora-alpha', '16',
    '--lr', '2e-4',
    '--batch-size', '2',
    '--epochs', '3',
]

# QLoRA command — adds --quantization int4
cmd_qlora = [
    'autotrain', 'llm',
    '--model', MODEL_NAME,
    '--data-path', DATA_PATH,
    '--train-split', 'train',
    '--project-name', PROJECT_QLORA,
    '--text-column', 'text',
    '--trainer', 'sft',
    '--peft',
    '--lora-r', '8',
    '--lora-alpha', '16',
    '--quantization', 'int4',   # QLoRA: 4-bit NF4 base + LoRA adapters
    '--lr', '2e-4',
    '--batch-size', '2',
    '--epochs', '3',
]

print('LoRA command:')
print('  ' + ' '.join(cmd_lora))
print()
print('QLoRA command (adds --quantization int4):')
print('  ' + ' '.join(cmd_qlora))

## Running Training via CLI

We run the standard LoRA command here. To switch to QLoRA, replace
`cmd_lora` with `cmd_qlora` in the cell below.

In [ ]:
print(f'Training: {MODEL_NAME} (LoRA, SFT)')
print(f'Output directory will be created at: ./{PROJECT_LORA}/')
print()

process = subprocess.Popen(
    cmd_lora,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env=os.environ.copy(),
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'AutoTrain failed (exit {process.returncode})')
print('\nTraining complete!')

## Launching the AutoTrain UI (No-Code Mode)

The UI provides a browser interface for configuring training jobs.
It is launched via the project's `start_autotrain_ui.sh` script or directly
from the cell below.

> **Gotcha #5 — The UI exposes fewer options than the CLI.**
> Quantization level, LoRA target modules, custom schedulers, and advanced
> trainer options are only available via CLI or Python API.
> Use the UI for quick prototyping; switch to CLI for production.

In [ ]:
import time

# Launches the AutoTrain UI at http://localhost:1236
# (port 1236 is forwarded by the devcontainer)
ui_env = os.environ.copy()
ui_process = subprocess.Popen(
    ['autotrain', 'app', '--host', '127.0.0.1', '--port', '1236'],
    env=ui_env,
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(3)

if ui_process.poll() is None:
    print('AutoTrain UI is running at http://localhost:1236')
    print('Run the NEXT cell to stop it when finished.')
else:
    err = ui_process.stderr.read().decode(errors='replace')
    print(f'UI failed to start:\n{err}')

In [ ]:
# Stop the UI
if 'ui_process' in dir() and ui_process.poll() is None:
    ui_process.terminate()
    print('AutoTrain UI stopped.')
else:
    print('UI is not running.')

## Understanding the Output Directory

After training, AutoTrain creates `<project-name>/` in the CWD:

```
autotrain_gpt2_lora/
├── adapter_model.safetensors   # LoRA weights
├── adapter_config.json          # PEFT configuration
├── tokenizer.json
├── tokenizer_config.json
├── training_params.json         # all hyperparameters used (keep for reproducibility)
├── autotrain.log                # training log
└── autotrain.db                 # internal AutoTrain state
```

`training_params.json` is especially useful: it records every flag used,
so you can reproduce the exact run later.

In [ ]:
output_dir = pathlib.Path(PROJECT_LORA)
if output_dir.exists():
    for f in sorted(output_dir.rglob('*')):
        rel = f.relative_to(output_dir)
        if f.is_file():
            print(f'  {rel}  ({f.stat().st_size:,} bytes)')
        else:
            print(f'  {rel}/')
else:
    print('Output directory not found — did training complete?')

## Summary

### Gotcha recap

| # | Gotcha | Fix |
|---|--------|-----|
| 1 | No `HF_TOKEN` → silent failures | Set both `HF_TOKEN` and `HUGGING_FACE_HUB_TOKEN` |
| 2 | Wrong column names in dataset | SFT needs exactly `text`; DPO needs `prompt`/`chosen`/`rejected` |
| 3 | Incorrect model ID | Use exact HuggingFace Hub ID (`gpt2`, not `GPT-2`) |
| 4 | Output dir created in CWD | Plan your CWD before running |
| 5 | UI has fewer options | Use CLI for quantization and advanced settings |

### LoRA / QLoRA CLI comparison

```bash
# LoRA
autotrain llm --model gpt2 --peft --lora-r 8 ...

# QLoRA (4-bit)
autotrain llm --model gpt2 --peft --lora-r 8 --quantization int4 ...

# 8-bit LoRA
autotrain llm --model gpt2 --peft --lora-r 8 --quantization int8 ...
```

**Next steps:** try `--trainer dpo` with a preference dataset, add
`--push-to-hub` to publish to the HuggingFace Hub, or switch to a larger
model like `facebook/opt-1.3b`.